# `Create Database for MIMIC Dataset`

Creates SQLite dataset for MIMIC-IV v3.1 

| **Done** | **Task**  |
|----------|-----------|
| [X] | Access Data Files |
| [X] | Design Databases |
| [X] | Read Aquisition Parameters |




## 1. `Define Settings`

### Workflow Annotation:

- Read the configuration file containing the path to the folder where the MIMIC dataset CSV files are stored.
- Load the settings from the configuration file.
- Verify and adjust the dataset path if necessary.
- Check the existence of all required CSV files in the specified folder.


In [ ]:
'''
Define the settings profile and load the project configuration.

Reads the project settings and builds the paths to all MIMIC CSV files
used in this notebook.
'''

from config.settings_data import DataSetsSettings
from utils import templates as T
from config.settings_data import load_csv_settings

profile = "default"

settings = DataSetsSettings.build( profile )
csv_settings = load_csv_settings( profile )



## 2.1 `Extract Imaging Data from MIMIC-CXR`


In [ ]:
'''
Generate image acquisition parameters.

Scans the MIMIC-CXR DICOM files and extracts the acquisition metadata
(view, kVp, detector type, exposure, pixel spacing, etc.) into the images
and image acquisition CSV files, so that it can be inserted into
the database.
'''

from utils.dataset_utils import generate_image_acquisition_parameter_csv

generate_image_acquisition_parameter_csv(
    settings.MIMIC_CXR_root,
    csv_settings.images_csv,
    csv_settings.image_acquisition_csv
    )  

In [ ]:
'''
Generate patient diagnosis data.

Formats the MIMIC-CXR-JPG CheXpert labels into a diagnosis CSV file
ready to be inserted into the database.
'''

from utils.dataset_utils import format_mimic_chexpert_diagnostic_csv

format_mimic_chexpert_diagnostic_csv(
    source_csv_path = csv_settings.source_chexpert_diagnosis_csv,
    target_csv_path = csv_settings.formated_chexpert_diagnosis_csv
    )

In [ ]:
'''
Generate study information.

Creates the studies CSV file, which links each imaging study (study_id)
to its patient (subject_id).
'''

from utils.dataset_utils import create_studies_csv

create_studies_csv(
    input_csv_path = csv_settings.source_chexpert_diagnosis_csv,
    output_csv_path = csv_settings.studies_csv
    )

In [ ]:
'''
Check that all required CSV files exist.

Note: the CSV file with the PIDs of patients missing from MIMIC-IV
(mimic_IV_missed_PID_list.csv) is generated later in this notebook,
so a "file not found" message for it is expected at this step.
'''

from utils.dataset_utils import check_MIMIC_IV_csv_file_existance

report = check_MIMIC_IV_csv_file_existance( csv_settings, T )

# 2.2 `Create Databese`



| **Done** | **Table Name**        | **Description** |
|-----------|-----------------------|-----------------|
| [X] | **omr**                | The Online Medical Record (OMR) table contains miscellaneous information from the EHR. |
| [X] | **provider**           | The provider table lists deidentified provider identifiers used in the database. |
| [X] | **admissions**         | Detailed information about hospital stays. |
| [X] | **d_hcpcs**            | Dimension table for hcpcsevents; provides a description of CPT codes. |
| [X] | **d_icd_diagnoses**    | Dimension table for diagnoses_icd; provides a description of ICD-9/ICD-10 billed diagnoses. |
| [X] | **d_icd_procedures**   | Dimension table for procedures_icd; provides a description of ICD-9/ICD-10 billed procedures. |
| [X] | **d_labitems**         | Dimension table for labevents; provides a description of all lab items. |
| [X] | **diagnoses_icd**      | Billed ICD-9/ICD-10 diagnoses for hospitalizations. |
| [X] | **drgcodes**           | Billed diagnosis related group (DRG) codes for hospitalizations. |
| [X] | **emar**               | The Electronic Medicine Administration Record (eMAR); barcode scanning of medications at the time of administration. |
| [X] | **emar_detail**        | Supplementary information for electronic administrations recorded in emar. |
| [X] | **hcpcsevents**        | Billed events occurring during the hospitalization. Includes CPT codes. |
| [X] | **labevents**          | Laboratory measurements sourced from patient derived specimens. |
| [X] | **microbiologyevents** | Microbiology cultures. |
| [X] | **patients**           | Patients' gender, age, and date of death if information exists. |
| [X] | **pharmacy**           | Formulary, dosing, and other information for prescribed medications. |
| [X] | **poe**                | Orders made by providers relating to patient care. |
| [X] | **poe_detail**         | Supplementary information for orders made by providers in the hospital. |
| [X] | **prescriptions**      | Prescribed medications. |
| [X] | **procedures_icd**     | Billed procedures for patients during their hospital stay. |
| [X] | **services**           | The hospital service(s) which cared for the patient during their hospitalization. |
| [X] | **transfers**          | Detailed information about patients' unit transfers. |

#### ICU Tables


| **Done** | **Table Name**        | **Description** |
|-----------|-----------------------|-----------------|
| [X] | **caregiver**                | The caregiver table lists deidentified provider identifiers used in the ICU module. |
| [X] | **d_items**           | Dimension table describing itemid. Defines concepts recorded in the events table in the ICU module. |
| [X] | **chartevents**         | Charted items occurring during the ICU stay. Contains the majority of information documented in the ICU. |
| [X] | **datetimeevents**            | Documented information which is in a date format (e.g. date of last dialysis). |
| [X] | **ICU stays**    | Tracking information for ICU stays including admission and discharge times. |
| [X] | **Ingredientevents**   | Ingredients of continuous or intermittent administrations including nutritional and water content. |
| [X] | **Inputevents**         | Information documented regarding continuous infusions or intermittent administrations. |
| [X] | **outputevents**      | Information regarding patient outputs including urine, drainage, and so on. |
| [X] | **procedureevents**           | Procedures documented during the ICU stay (e.g. ventilation), though not necessarily conducted within the ICU (e.g. x-ray imaging). |


In [ ]:
'''
Create the database.

Creates an SQLite database with the specified name at the specified
location, using the existing SQL schema script.
'''

from utils.dataset_utils import create_database_from_sql

create_database_from_sql(
    db_path = settings.DB_name,
    schema_path = settings.SQL_script
)

# 2.3 `Data Correction`


In [ ]:
'''
Remove records that violate table constraints.

Deletes rows that would violate the database table constraints
(e.g., NOT NULL) from the source CSV files before they are inserted
into the database. The CSV files are modified in place; a .bak backup
is created.
'''

from utils.mimic_dataset_corrections import clean_microbiologyevents_csv_inplace, clean_prescriptions_required_columns_inplace

clean_microbiologyevents_csv_inplace(
    csv_settings.microbiologyevents_csv
)

clean_prescriptions_required_columns_inplace(
    csv_settings.prescriptions_csv
)


There are 3511 missing PIDs


# 2.4 `Missing PIDs Correction`

In [ ]:
'''
Insert placeholder records for missing patients.

Identifies patient IDs (PIDs) that are present in MIMIC-CXR but missing
from MIMIC-IV, and inserts placeholder records for them into the patients
table (only subject_id is filled, other columns are NULL).

Note: must be run before the data insertion step, because the studies
table references patients(subject_id) and foreign keys are enforced.
'''

from utils.mimic_dataset_corrections import check_csv_study_id_match, insert_missing_subject_ids

missing_cxr_pid = check_csv_study_id_match(
    csv_settings.patients_csv,
    csv_settings.studies_csv,
    csv_settings.missing_cxr_pid_csv
)

insert_missing_subject_ids(
    settings.DB_name,
    csv_settings.missing_cxr_pid_csv
)

# 3. `Inject Data`


In [ ]:
'''
Insert data into the database.

Defines the order in which the tables are filled (parent tables first,
to satisfy foreign keys) and runs the function that writes the data
from the CSV files into each table.
'''

from utils.dataset_utils import get_csv_table_mapping, inject_csv_to_table

TABLE_INSERT_ORDER, TABLE_TO_CSV = get_csv_table_mapping( csv_settings )

for table_name in TABLE_INSERT_ORDER:
    print('Processing: '+table_name)

    csv_path = TABLE_TO_CSV[table_name]
    inject_csv_to_table(
        settings.DB_name,
        csv_path,
        table_name
    )


Processing: patients
Processing: provider
Processing: d_hcpcs
Processing: d_icd_diagnoses
Processing: d_icd_procedures
Processing: d_labitems
Processing: admissions
Processing: omr
Processing: hcpcsevents
Processing: diagnoses_icd
Processing: procedures_icd
Processing: labevents
Processing: microbiologyevents
Processing: drgcodes
Processing: services
Processing: transfers
Processing: poe
Processing: poe_detail
Processing: pharmacy
Processing: prescriptions
Processing: emar
Processing: emar_detail
Processing: studies
Processing: images
Processing: image_acquisition
Processing: chexpert_diagnosis


Injection issues:

1. NOT NULL constraint failed: microbiologyevents.spec_type_desc ->
Backup created: D:\003.Data\MIMIC-IV.v3.1\physionet.org\files\mimiciv\3.1\hosp\microbiologyevents.csv.bak
Cleaned file overwritten: D:\003.Data\MIMIC-IV.v3.1\physionet.org\files\mimiciv\3.1\hosp\microbiologyevents.csv
Total rows: 3988224
Removed rows: 1
Kept rows: 3988223

2. NOT NULL constraint failed: prescriptions.drug ->

3. MIMIC-IV does not provide information for 3511 patients in MIMIC-CXR 

